# Query Rewriting + LLM Reranking

Loads the existing `aethon_kb` ChromaDB collection and adds two layers on top:

| Layer | What it does | Why |
|---|---|---|
| **Query rewriting** | Expands the user's question before retrieval | Raw questions embed poorly — rewritten queries surface better chunks |
| **LLM reranking** | Scores every retrieved chunk against the original question | Cosine similarity ≠ relevance — LLM catches semantic mismatches |

Two rewrite strategies implemented: **HyDE** (hypothetical answer) and **multi-query** (paraphrases).  
Ablation table at the end compares all three modes side by side.

## 0 — Setup

In [1]:
import os, json, logging
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

logging.getLogger("chromadb.telemetry").setLevel(logging.ERROR)

load_dotenv(dotenv_path=os.path.join("..", ".env"))
oai = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

CHROMA_PATH     = "./chroma_db"
COLLECTION_NAME = "aethon_kb"
EMBED_MODEL     = "text-embedding-3-small"
LLM_MODEL       = "gpt-4o-mini"
TOP_K           = 10    # chunks to pull from Chroma before reranking
RERANK_K        = 3     # chunks to keep after reranking

# Connect to existing collection
chroma     = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma.get_collection(name=COLLECTION_NAME)
print(f"Collection '{COLLECTION_NAME}' — {collection.count()} chunks ready")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Collection 'aethon_kb' — 31 chunks ready


## 1 — Embedding helper

In [2]:
def embed(text: str) -> list[float]:
    """Single-text embed. Returns a unit-normalised 1536-dim vector."""
    vec = oai.embeddings.create(input=[text], model=EMBED_MODEL).data[0].embedding
    v   = np.array(vec, dtype=np.float32)
    return (v / np.linalg.norm(v)).tolist()

## 2 — Base retriever (no rewriting, no reranking)
Baseline to compare against.

In [3]:
def retrieve_base(question: str, n: int = TOP_K) -> list[dict]:
    """
    Plain cosine retrieval — embed the raw question, pull top-n from Chroma.
    Returns list of {rank, chunk_id, topic, filename, score, text}.
    """
    results = collection.query(
        query_embeddings=[embed(question)],
        n_results=n,
        include=["documents", "metadatas", "distances"],
    )
    hits = []
    for i, (doc, meta, dist) in enumerate(
        zip(results["documents"][0], results["metadatas"][0], results["distances"][0])
    ):
        hits.append({
            "rank":     i + 1,
            "chunk_id": results["ids"][0][i],
            "topic":    meta["topic"],
            "filename": meta["filename"],
            "score":    round(1 - dist, 4),   # chroma returns distance; convert to similarity
            "text":     doc,
        })
    return hits

## 3 — Query rewriting

### 3a — HyDE (Hypothetical Document Embedding)
Generate a *hypothetical answer* to the question, then embed that instead of the question.  
**Why it works:** answers embed much closer to answer-bearing chunks than questions do — different vocabulary, same semantic space.

In [4]:
def hyde(question: str) -> str:
    """
    Generate a hypothetical passage that would answer the question.
    We embed THIS instead of the raw question.
    """
    resp = oai.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content":
             "Write a short factual passage (3-5 sentences) that directly answers "
             "the question. Write as if you are certain — no hedging. "
             "Use the kind of language that would appear in a company knowledge base."},
            {"role": "user", "content": question},
        ],
        temperature=0,
        max_tokens=200,
    )
    return resp.choices[0].message.content.strip()


def retrieve_hyde(question: str, n: int = TOP_K) -> list[dict]:
    """Retrieve using the HyDE passage instead of the raw question."""
    hypo = hyde(question)
    results = collection.query(
        query_embeddings=[embed(hypo)],
        n_results=n,
        include=["documents", "metadatas", "distances"],
    )
    hits = []
    for i, (doc, meta, dist) in enumerate(
        zip(results["documents"][0], results["metadatas"][0], results["distances"][0])
    ):
        hits.append({
            "rank":      i + 1,
            "chunk_id":  results["ids"][0][i],
            "topic":     meta["topic"],
            "filename":  meta["filename"],
            "score":     round(1 - dist, 4),
            "text":      doc,
            "hyde_used": hypo,     # keep for inspection
        })
    return hits

### 3b — Multi-query
Generate N paraphrases of the question, retrieve for each, union the results.  
**Why it works:** different phrasings surface different chunks — boosts recall at the cost of more API calls.

In [5]:
def multi_query(question: str, n_variants: int = 3) -> list[str]:
    """Return the original question + n_variants paraphrases."""
    resp = oai.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content":
             f"Generate {n_variants} alternative phrasings of the search query. "
             "Vary vocabulary and specificity. One per line, no numbering, no bullets."},
            {"role": "user", "content": question},
        ],
        temperature=0.4,
        max_tokens=200,
    )
    variants = [
        line.strip() for line in resp.choices[0].message.content.splitlines()
        if line.strip()
    ]
    return [question] + variants[:n_variants]


def retrieve_multi(question: str, n: int = TOP_K) -> list[dict]:
    """
    Retrieve for each query variant, union results (dedup on chunk_id),
    keep the best score per chunk.
    """
    queries  = multi_query(question)
    pool: dict[str, dict] = {}

    for q in queries:
        results = collection.query(
            query_embeddings=[embed(q)],
            n_results=n,
            include=["documents", "metadatas", "distances"],
        )
        for cid, doc, meta, dist in zip(
            results["ids"][0], results["documents"][0],
            results["metadatas"][0], results["distances"][0]
        ):
            sim = round(1 - dist, 4)
            if cid not in pool or sim > pool[cid]["score"]:
                pool[cid] = {
                    "chunk_id": cid,
                    "topic":    meta["topic"],
                    "filename": meta["filename"],
                    "score":    sim,
                    "text":     doc,
                }

    ranked = sorted(pool.values(), key=lambda x: x["score"], reverse=True)
    for i, h in enumerate(ranked):
        h["rank"] = i + 1
    return ranked

## 4 — LLM reranking

Takes the top-k candidates from ANY retriever above and asks the LLM to score each chunk 0-10 for relevance to the **original** question (not the rewritten one).  

Uses **structured outputs** so scores are guaranteed integers — no parsing fragility.

In [6]:
RERANK_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "rerank_scores",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "scores": {
                    "type": "array",
                    "description": "One score per passage, in the same order as the input.",
                    "items": {
                        "type": "object",
                        "properties": {
                            "index":     {"type": "integer", "description": "0-based index of the passage."},
                            "score":     {"type": "integer", "description": "Relevance 0-10. 10=perfectly answers the question, 0=completely irrelevant."},
                            "reasoning": {"type": "string",  "description": "One sentence explaining the score."}
                        },
                        "required": ["index", "score", "reasoning"],
                        "additionalProperties": False
                    }
                }
            },
            "required": ["scores"],
            "additionalProperties": False
        }
    }
}


def llm_rerank(question: str, candidates: list[dict], top_k: int = RERANK_K) -> list[dict]:
    """
    Score each candidate 0-10 for relevance to the original question.
    Returns top_k candidates sorted by LLM score descending.
    """
    # Build the passage listing
    listing = "\n\n".join(
        f"[{i}] {c['text'][:400]}"
        for i, c in enumerate(candidates)
    )
    prompt = (
        f"QUESTION: {question}\n\n"
        f"Score each passage 0-10 for how well it answers the question.\n\n"
        f"{listing}"
    )
    resp = oai.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content":
             "You are a relevance judge. Score each passage strictly by how directly "
             "it answers the question. 10 = directly answers it, 0 = completely off-topic."},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
        max_tokens=1024,
        response_format=RERANK_SCHEMA,
    )
    scored = json.loads(resp.choices[0].message.content)["scores"]

    # Attach LLM scores back to candidates
    for s in scored:
        candidates[s["index"]]["llm_score"]     = s["score"]
        candidates[s["index"]]["llm_reasoning"] = s["reasoning"]

    reranked = sorted(candidates, key=lambda x: x.get("llm_score", 0), reverse=True)
    for i, h in enumerate(reranked):
        h["rerank"] = i + 1
    return reranked[:top_k]

## 5 — Display helper

In [7]:
def show(label: str, question: str, hits: list[dict], show_reasoning: bool = False):
    print(f"{'='*60}")
    print(f" {label}")
    print(f" Q: {question}")
    print(f"{'='*60}")
    for h in hits:
        rank  = h.get('rerank', h.get('rank', '?'))
        score = f"cosine={h['score']:.4f}"
        if 'llm_score' in h:
            score += f"  llm={h['llm_score']}/10"
        print(f"  [{rank}] {h['topic']}  ({h['filename']})")
        print(f"       {score}")
        if show_reasoning and 'llm_reasoning' in h:
            print(f"       reason: {h['llm_reasoning']}")
        print(f"       {h['text'][:180]}...")
        print()

## 6 — Run all three modes + reranking

For each question we run:
1. **Base** — raw question, cosine only
2. **HyDE** — hypothetical answer, cosine only  
3. **Multi-query** — 3 paraphrases, union + cosine
4. **Multi-query + LLM rerank** — best of multi-query, then LLM-scored

In [8]:
questions = [
    "Who leads the engineering team at Aethon?",
    "How does VoltCore make money?",
    "What awards has Aethon won?",
]

for q in questions:
    base   = retrieve_base(q,  n=RERANK_K)
    hyde_r = retrieve_hyde(q,  n=RERANK_K)
    multi  = retrieve_multi(q, n=TOP_K)
    reranked = llm_rerank(q, multi[:TOP_K], top_k=RERANK_K)

    show("BASE (raw question, cosine)",          q, base)
    show("HYDE (hypothetical answer, cosine)",   q, hyde_r)
    show("MULTI-QUERY (union, cosine)",          q, multi[:RERANK_K])
    show("MULTI-QUERY + LLM RERANK",             q, reranked, show_reasoning=True)
    print()

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


 BASE (raw question, cosine)
 Q: Who leads the engineering team at Aethon?
  [1] Document Overview  (employees.md)
       cosine=0.6826
       # Aethon Dynamics — Employees

This document covers the leadership team and selected key employees at Aethon Dynamics....

  [2] Company Structure  (company_overview.md)
       cosine=0.6724
       ## Structure

Aethon is organized into four divisions: Engineering, Product, Go-to-Market (Sales & Marketing), and Operations. Each division is led by a VP who reports to the CEO. ...

  [3] Dr. Maya Krishnan  (employees.md)
       cosine=0.6611
       ## Leadership Team

### Dr. Maya Krishnan — Chief Executive Officer
Maya co-founded Aethon Dynamics in 2016. Before Aethon, she spent eight years as a power systems engineer at a r...

 HYDE (hypothetical answer, cosine)
 Q: Who leads the engineering team at Aethon?
  [1] Company Structure  (company_overview.md)
       cosine=0.6837
       ## Structure

Aethon is organized into four divisions: Engineeri

## 7 — Ablation table

Compares which chunk landed at rank 1 across all four modes for each question.  
This is the qualitative ablation — the quantitative version (MRR, recall@k) comes in the eval notebook.

In [9]:
print(f"{'Question':<45} {'Mode':<25} {'Rank-1 topic'}")
print("-" * 110)

for q in questions:
    base     = retrieve_base(q,  n=RERANK_K)
    hyde_r   = retrieve_hyde(q,  n=RERANK_K)
    multi    = retrieve_multi(q, n=TOP_K)
    reranked = llm_rerank(q, multi[:TOP_K], top_k=RERANK_K)

    rows = [
        ("Base",                  base[0]["topic"]),
        ("HyDE",                  hyde_r[0]["topic"]),
        ("Multi-query",           multi[0]["topic"]),
        ("Multi-query + rerank",  reranked[0]["topic"]),
    ]
    q_short = q[:43] + ".." if len(q) > 43 else q
    for i, (mode, topic) in enumerate(rows):
        q_col = q_short if i == 0 else ""
        print(f"{q_col:<45} {mode:<25} {topic}")
    print()

Question                                      Mode                      Rank-1 topic
--------------------------------------------------------------------------------------------------------------
Who leads the engineering team at Aethon?     Base                      Document Overview
                                              HyDE                      Company Structure
                                              Multi-query               Document Overview
                                              Multi-query + rerank      Daniel Osei

How does VoltCore make money?                 Base                      VoltCore Product Details
                                              HyDE                      VoltCore Product Details
                                              Multi-query               VoltCore Product Details
                                              Multi-query + rerank      VoltCore Product Details

What awards has Aethon won?                   Base          